In [ ]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 8.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Bibliothèque**

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import json
from sentence_transformers import util
import ast
from IPython.display import display
from sklearn.metrics import precision_score, recall_score, f1_score
from datasets import load_dataset

/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


**Charger le dataset**

In [ ]:
dataset = load_dataset("THUIR/AEOLLM", split='train')
df = pd.DataFrame(dataset)

df.head()

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


,taskId,taskName,questionId,question,answerId,answer,score,rank
0,3,NFCATS,0,Are constructed languages useful and do we nee...,0,"Constructed languages, also known as conlangs,...",5,1
1,3,NFCATS,0,Are constructed languages useful and do we nee...,1,Here is a 187-word answer to the question:\n\...,4,3
2,3,NFCATS,0,Are constructed languages useful and do we nee...,2,"Constructed languages, also known as artificia...",4,3
3,3,NFCATS,0,Are constructed languages useful and do we nee...,3,"Constructed languages, such as English and Fre...",4,3
4,3,NFCATS,0,Are constructed languages useful and do we nee...,4,Constructed languages are a group of artificia...,4,3


In [ ]:
# Charger le dataset
df = pd.read_csv('/content/drive/MyDrive/NTCIR_dataset.csv')
# Pour chaque questionId, mettre à jour les questions pour garder uniquement la première
df['question'] = df.groupby('questionId')['question'].transform('first')

df.to_csv('/content/drive/MyDrive/NTCIR_dataset_updated.csv', index=False)

# Afficher le résultat
df

,taskId,taskName,questionId,question,answerId,answer,score,rank
0,3,NFCATS,0,Are constructed languages useful and do we nee...,0,"Constructed languages, also known as conlangs,...",5,1
1,3,NFCATS,0,Are constructed languages useful and do we nee...,1,Here is a 187-word answer to the question:\n\...,4,3
2,3,NFCATS,0,Are constructed languages useful and do we nee...,2,"Constructed languages, also known as artificia...",4,3
3,3,NFCATS,0,Are constructed languages useful and do we nee...,3,"Constructed languages, such as English and Fre...",4,3
4,3,NFCATS,0,Are constructed languages useful and do we nee...,4,Constructed languages are a group of artificia...,4,3
...,...,...,...,...,...,...,...,...
555,1,story,19,what does TT mean in motorcycle racing?,555,"In the vast expanse of space, the man felt utt...",3,1
556,1,story,19,what does TT mean in motorcycle racing?,556,"The man was lost in space, with no fuel, no fo...",2,6
557,1,story,19,what does TT mean in motorcycle racing?,557,"The man, a space explorer named John, had been...",2,6
558,1,story,19,what does TT mean in motorcycle racing?,558,"David floated in the vast, unending void, the ...",3,1


**Génération des embeddings**

In [ ]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Extraire les colonnes de texte (question et réponse)
questions = df['question'].tolist()
answers = df['answer'].tolist()

# Générer les embeddings pour les questions et les réponses
question_embeddings = model.encode(questions, show_progress_bar=True, convert_to_tensor=False)  # Convert to numpy array
answer_embeddings = model.encode(answers, show_progress_bar=True, convert_to_tensor=False)  # Convert to numpy array

# Convertir les embeddings en listes pour les stocker dans le DataFrame
df['question_embeddings'] = [json.dumps(embedding.tolist()) for embedding in question_embeddings]
df['answer_embeddings'] = [json.dumps(embedding.tolist()) for embedding in answer_embeddings]

# Afficher les premières lignes pour vérifier
df.head()

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

,taskId,taskName,questionId,question,answerId,answer,score,rank,question_embeddings,answer_embeddings
0,3,NFCATS,0,Are constructed languages useful and do we nee...,0,"Constructed languages, also known as conlangs,...",5,1,"[-0.024295108392834663, 0.0030533275566995144,...","[-0.01251829694956541, -0.006497832480818033, ..."
1,3,NFCATS,0,Are constructed languages useful and do we nee...,1,Here is a 187-word answer to the question:\n\...,4,3,"[-0.024295108392834663, 0.0030533275566995144,...","[0.03790402412414551, -0.024686668068170547, 0..."
2,3,NFCATS,0,Are constructed languages useful and do we nee...,2,"Constructed languages, also known as artificia...",4,3,"[-0.024295108392834663, 0.0030533275566995144,...","[0.030249370262026787, -0.03951647877693176, 0..."
3,3,NFCATS,0,Are constructed languages useful and do we nee...,3,"Constructed languages, such as English and Fre...",4,3,"[-0.024295108392834663, 0.0030533275566995144,...","[-0.006921980995684862, -0.002302184933796525,..."
4,3,NFCATS,0,Are constructed languages useful and do we nee...,4,Constructed languages are a group of artificia...,4,3,"[-0.024295108392834663, 0.0030533275566995144,...","[-0.005974634550511837, -0.030247077345848083,..."


In [ ]:
output_file_path = '/content/drive/My Drive/NTCIR__embeddings.csv'
df.to_csv(output_file_path, index=False)
print(f"File saved with embeddings at: {output_file_path}")

File saved with embeddings at: /content/drive/My Drive/NTCIR__embeddings.csv


In [ ]:
# Load the CSV file with embeddings
file_path = '/content/drive/My Drive/NTCIR__embeddings.csv'
loaded_df  = pd.read_csv(file_path)
loaded_df

,taskId,taskName,questionId,question,answerId,answer,score,rank,question_embeddings,answer_embeddings
0,3,NFCATS,0,Are constructed languages useful and do we nee...,0,"Constructed languages, also known as conlangs,...",5,1,"[-0.024295108392834663, 0.0030533275566995144,...","[-0.01251829694956541, -0.006497832480818033, ..."
1,3,NFCATS,0,Are constructed languages useful and do we nee...,1,Here is a 187-word answer to the question:\n\...,4,3,"[-0.024295108392834663, 0.0030533275566995144,...","[0.03790402412414551, -0.024686668068170547, 0..."
2,3,NFCATS,0,Are constructed languages useful and do we nee...,2,"Constructed languages, also known as artificia...",4,3,"[-0.024295108392834663, 0.0030533275566995144,...","[0.030249370262026787, -0.03951647877693176, 0..."
3,3,NFCATS,0,Are constructed languages useful and do we nee...,3,"Constructed languages, such as English and Fre...",4,3,"[-0.024295108392834663, 0.0030533275566995144,...","[-0.006921980995684862, -0.002302184933796525,..."
4,3,NFCATS,0,Are constructed languages useful and do we nee...,4,Constructed languages are a group of artificia...,4,3,"[-0.024295108392834663, 0.0030533275566995144,...","[-0.005974634550511837, -0.030247077345848083,..."
...,...,...,...,...,...,...,...,...,...,...
555,1,story,19,what does TT mean in motorcycle racing?,555,"In the vast expanse of space, the man felt utt...",3,1,"[-0.018420632928609848, 0.06285811215639114, -...","[-0.02878694050014019, 0.03584077209234238, 0...."
556,1,story,19,what does TT mean in motorcycle racing?,556,"The man was lost in space, with no fuel, no fo...",2,6,"[-0.018420632928609848, 0.06285811215639114, -...","[-0.04499581828713417, 0.08706449717283249, 0...."
557,1,story,19,what does TT mean in motorcycle racing?,557,"The man, a space explorer named John, had been...",2,6,"[-0.018420632928609848, 0.06285811215639114, -...","[-0.051828570663928986, 0.051165997982025146, ..."
558,1,story,19,what does TT mean in motorcycle racing?,558,"David floated in the vast, unending void, the ...",3,1,"[-0.018420632928609848, 0.06285811215639114, -...","[0.07022567093372345, 0.07191848754882812, 0.0..."


**Calcule de similarité et récupérer les réponses les plus pertinentes pour chaque question**

In [ ]:
# Function to load embeddings
def load_embeddings(embedding_list):
    return np.array([json.loads(embedding) for embedding in embedding_list])

# Charger les embeddings
question_embeddings = load_embeddings(loaded_df['question_embeddings'])
answer_embeddings = load_embeddings(loaded_df['answer_embeddings'])


In [ ]:
# Calcul de la similarité cosinus entre les réponses et les questions
cos_sim_matrix = cosine_similarity(answer_embeddings, question_embeddings)

# Liste pour stocker les résultats
results = []

# Sélectionner les 30 réponses les plus pertinentes pour chaque question
for i in range(cos_sim_matrix.shape[1]):  # Itérer sur chaque question
    # Trier les indices des réponses en fonction de la similarité pour la question i (tri décroissant)
    relevant_answers_indices = np.argsort(cos_sim_matrix[:, i])[::-1][:30]  # Tri décroissant

    # Sélectionner seulement les 30 premières réponses
    top_30_indices = relevant_answers_indices[:30]

    # Pour chaque réponse sélectionnée, récupérer les informations et les stocker
    for answer_idx in top_30_indices:
        answer = loaded_df.iloc[answer_idx]  # Réponse correspondante
        question = loaded_df.iloc[i]  # Question correspondante

        score = answer['score']
        # Ajouter les informations dans la liste des résultats
        results.append({
            'question_id': question['questionId'],
            'question_text': question['question'],
            'answer_id': answer['answerId'],
            'answer_text': answer['answer'],
            'score': score,
            'cosine_similarity': cos_sim_matrix[answer_idx][i]  # Ajouter la similarité cosinus
        })

# Créer un DataFrame avec les résultats
retrieved_docs_top_30 = pd.DataFrame(results)

# Trier les résultats par la similarité cosinus (en ordre décroissant)
retrieved_docs_top_30 = retrieved_docs_top_30.sort_values(by='cosine_similarity', ascending=False)

# Afficher un aperçu des résultats
retrieved_docs_top_30


,question_id,question_text,answer_id,answer_text,score,cosine_similarity
10740,11,Should the Tampon Tax be Abolished?,82,Absolutely. The Tampon Tax should absolutely b...,5,0.903187
2490,11,Should the Tampon Tax be Abolished?,82,Absolutely. The Tampon Tax should absolutely b...,5,0.903187
10710,11,Should the Tampon Tax be Abolished?,82,Absolutely. The Tampon Tax should absolutely b...,5,0.903187
2460,11,Should the Tampon Tax be Abolished?,82,Absolutely. The Tampon Tax should absolutely b...,5,0.903187
2430,11,Should the Tampon Tax be Abolished?,82,Absolutely. The Tampon Tax should absolutely b...,5,0.903187
...,...,...,...,...,...,...
3569,16,Why Photoshop CC is printing only in B&W?,27,The music industry can be complex when it come...,5,0.107060
16049,16,Why Photoshop CC is printing only in B&W?,27,The music industry can be complex when it come...,5,0.107060
7589,16,Why Photoshop CC is printing only in B&W?,27,The music industry can be complex when it come...,5,0.107060
7619,16,Why Photoshop CC is printing only in B&W?,27,The music industry can be complex when it come...,5,0.107060


In [ ]:
# Définir un seuil pour déterminer la pertinence
threshold = 4

# Créer un dictionnaire pour stocker le nombre de documents pertinents par query_id
relevant_retrieved_in_top30 = {}

# Créer un dictionnaire pour stocker les pertinences binaires (0 ou 1) pour chaque query_id
relevant_binary_dict = {}

# Parcourir chaque ligne de retrieved_docs_top_30 (chaque document récupéré)
for i, row in retrieved_docs_top_30.iterrows():
    document_id = row['answer_id']  # On suppose que l'ID du document est 'answer_id'
    query_id = row['question_id']
    score = row['score']

    # Vérifier si le score est supérieur ou égal au seuil de pertinence
    is_relevant = 1 if score >= threshold else 0

    # Si le query_id n'existe pas encore dans le dictionnaire, l'initialiser
    if query_id not in relevant_retrieved_in_top30:
        relevant_retrieved_in_top30[query_id] = 0

    # Incrémenter le compteur de documents pertinents pour cette requête si le document est pertinent
    if is_relevant == 1:
        relevant_retrieved_in_top30[query_id] += 1

    # Ajouter la pertinence binaire dans le dictionnaire
    if query_id not in relevant_binary_dict:
        relevant_binary_dict[query_id] = []

    # Limiter à 30 résultats
    if len(relevant_binary_dict[query_id]) < 30:
        relevant_binary_dict[query_id].append(is_relevant)

# Trier les query_ids par le nombre de documents pertinents, en ordre décroissant
sorted_query_ids = sorted(relevant_retrieved_in_top30.items(), key=lambda x: x[1], reverse=True)

# Prendre seulement les 30 premiers query_id
top_30_query_ids = sorted_query_ids[:30]

# Afficher le nombre de documents pertinents pour les 30 premiers query_id triés
print("Nombre de documents pertinents pour les 30 premiers query_id triés :")
for query_id, _ in top_30_query_ids:
    # Calculer la somme des pertinences binaires pour chaque query_id
    relevant_count = sum(relevant_binary_dict[query_id])
    print(f"{query_id}: {relevant_count}")


Nombre de documents pertinents pour les 30 premiers query_id triés :
2: 30
6: 30
11: 30
4: 2
9: 30
10: 30
5: 30
17: 30
0: 30
15: 30
1: 2
3: 28
13: 30
16: 2
18: 2
7: 0
8: 30
19: 30
14: 28
12: 2


In [ ]:
# Afficher les pertinences binaires pour les 30 premiers query_id triés
print("\nPertinences binaires pour les 30 premiers query_id triés :")
for query_id, _ in top_30_query_ids:
    print(f"{query_id}: {relevant_binary_dict[query_id]}")


Pertinences binaires pour les 30 premiers query_id triés :
2: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
6: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
11: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
4: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1]
9: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
10: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
5: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
17: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
0: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
15: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

**Calculer les métriques (Précision, Rappel, F1, MAP)**

In [ ]:
# Fonction pour calculer P@k
def precision_at_k(relevant_binary_dict, k=30):
    p_at_k = {}
    for query_id, relevant_list in relevant_binary_dict.items():
        relevant_count = sum(relevant_list[:k])
        p_at_k[query_id] = relevant_count / k
    return p_at_k

# Fonction pour calculer R@k
def recall_at_k(relevant_binary_dict, total_relevant_docs, k=30):
    r_at_k = {}
    for query_id, relevant_list in relevant_binary_dict.items():
        relevant_retrieved = sum(relevant_list[:k])
        total_relevant = total_relevant_docs.get(query_id, 0)
        r_at_k[query_id] = relevant_retrieved / total_relevant if total_relevant > 0 else 0
    return r_at_k

# Fonction pour calculer F1@k
def f1_at_k(p_at_k, r_at_k):
    f1_at_k = {}
    for query_id in p_at_k:
        precision = p_at_k.get(query_id, 0)
        recall = r_at_k.get(query_id, 0)
        f1_at_k[query_id] = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return f1_at_k

# Fonction pour calculer l'Average Precision (AP) pour une requête donnée
def average_precision(relevant_retrieved_list):
    relevant_count = 0
    precision_sum = 0
    for i, relevance in enumerate(relevant_retrieved_list):
        if relevance == 1:
            relevant_count += 1
            precision_sum += relevant_count / (i + 1)
    return precision_sum / relevant_count if relevant_count > 0 else 0

# Fonction pour calculer la MAP (Mean Average Precision)
def mean_average_precision(relevant_retrieved_in_top_k):
    # Calculer l'AP pour chaque requête
    ap_list = [average_precision(relevant_retrieved_list) for relevant_retrieved_list in relevant_retrieved_in_top_k.values()]
    # Calculer la moyenne des AP pour obtenir le MAP
    return sum(ap_list) / len(ap_list) if ap_list else 0


In [ ]:
# Calculer P@30
p_at_k = precision_at_k(relevant_binary_dict)

# Calculer R@30
r_at_k = recall_at_k(relevant_binary_dict, relevant_retrieved_in_top30)

# Calculer F1@30
f1_at_k_score = f1_at_k(p_at_k, r_at_k)

# Calculer MAP (Mean Average Precision)
ap_list = [average_precision(relevant_binary_dict[q]) for q in relevant_binary_dict]
map_score = mean_average_precision(relevant_binary_dict)

# Créer un DataFrame pour afficher les résultats
df_results = pd.DataFrame({
    'query_id': list(p_at_k.keys()),
    'P@30': list(p_at_k.values()),
    'R@30': list(r_at_k.values()),
    'F1@30': list(f1_at_k_score.values()),
    'AP': ap_list
})

# Afficher les résultats
df_results

,query_id,P@30,R@30,F1@30,AP
0,11,1.000000,0.042857,0.082192,1.000000
1,16,0.066667,0.003759,0.007117,0.050575
2,10,1.000000,0.046584,0.089021,1.000000
3,4,0.066667,0.002857,0.005479,0.050575
4,5,1.000000,0.046584,0.089021,1.000000
5,8,1.000000,0.066964,0.125523,1.000000
6,3,0.933333,0.050000,0.094915,1.000000
7,13,1.000000,0.053571,0.101695,1.000000
8,17,1.000000,0.048701,0.092879,1.000000
9,0,1.000000,0.051020,0.097087,1.000000


In [ ]:
print(f"MAP Score : {map_score}")

MAP Score : 0.7126436781609196
